# DATA-445
## Exam 1 - Part 1

### Instructions

- **Exam 1 - Part 1** is worth **96** points.
- Please submit a **.ipynb** file to Blackboard.
- **Please strive for clarity and organization.**
- **AI usage**: 
   - You may use AI exclusively for **debugging**. 
   - **Do not** use AI to generate answers.
- **Due Date: October 13, 2025, by 12:50 PM.**

## Exercise 1

Consider the `train_df.csv` datafile. This dataset contains information about students' demographics, study habits, and learning environment, together with their performance on an exam. The goal is to build regression models to predict `exam_score`.

**(a)** (2 points) Read the datafile and create a data-frame called `df`. Drop the `id` column since it is only a row identifier and carries no predictive information.

In [18]:
import pandas as pd

df = pd.read_csv('train_df.csv')

df = df.dropna()

df.head()


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,364426,18,other,b.sc,3.07,88.3,yes,4.1,average,coaching,low,hard,51.3
1,224752,21,male,b.com,3.28,49.6,no,7.0,poor,online videos,high,moderate,50.6
2,110423,20,female,bca,4.45,42.5,yes,6.7,good,group study,high,moderate,79.9
3,272555,21,male,b.com,4.19,82.8,yes,5.7,poor,mixed,low,hard,55.4
4,199651,21,female,b.com,1.31,91.2,yes,8.5,poor,mixed,low,easy,50.4


**(b)** (4 points) There is one boolean-like column: `internet_access` (`yes`/`no`). Convert `no` to 0 and `yes` to 1.

In [19]:
df['internet_access'] = df['internet_access'].str.lower().map({'no': 0, 'yes': 1})

**(c)** (8 points) There are six object columns: `gender`, `course`, `sleep_quality`, `study_method`, `facility_rating`, and `exam_difficulty`. Convert those labels into numbers. *Hint*: you may consider using `OrdinalEncoder` from `sklearn.preprocessing`.

In [20]:
from sklearn.preprocessing import OrdinalEncoder

columns = ['gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']
encoder = OrdinalEncoder()
df[columns] = encoder.fit_transform(df[columns])


**(d)** (4 points) Define `exam_score` as the target variable, and the other variables as the input features. Also, define the 5-fold cross-validation strategy.

In [21]:
from sklearn.model_selection import StratifiedKFold

x = df[['gender', 'course', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']]
y = df['exam_score']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

**(e)** (15 points) Train and evaluate the `Ridge` regression model over the 5-fold cross-validation strategy defined in part (d). Ensure to standardize the inputs and optimize the value of `alpha` first and then run the cross-validation experiment. Use the root mean squared error as measure of performance.

In [22]:
from sklearn.pipeline import Pipeline



**(f)** (15 points) Train and evaluate the `RandomForestRegressor` model (`n_estimators=1000`, `max_depth=10`, and `random_state=42`) over the 5-fold cross-validation strategy defined in part (d). Use the root mean squared error as measure of performance.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold

rf = RandomForestRegressor(n_estimators=1000, max_depth=10, random_state=42, n_jobs=-1)

scores = -cross_val_score(rf, x, y, cv=skf, scoring="neg_root_mean_squared_error")

print("RMSE per fold:", np.round(scores, 4))
print(f"Mean RMSE: {scores.mean():.4f} (± {scores.std():.4f})")

NameError: name 'cv' is not defined

**(g)** (15 points) Train and evaluate the `MLPRegressor` with two hidden layers: the first one with 8 neurons and `ReLU` as the activation function, the second one with 8 neurons and `ReLU` as the activation function. Also use `solver='adam'`, `max_iter=200` and `batch_size=256`, over the 5-fold cross-validation strategy defined in part (d). Use the root mean squared error as measure of performance. Ensure to standardize the input features before you build the model.

In [28]:
from sklearn.model_selection import cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import recall_score

mlp_relu = make_pipeline(
    MinMaxScaler(),
    MLPClassifier(hidden_layer_sizes=(4,),
                 activation='relu',
                 max_iter=200,
                 solver="adam",   
                 batch_size=256,      
                 random_state=42)
)

recall_scores = []
for i, (train_index, test_index) in enumerate(skf.split(x, y)):
    
    x_train, x_test = x.iloc[train_index], x.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    mlp_relu.fit(x_train, y_train)
    relu_pred = mlp_relu.predict_proba(x_test)[:, 1]
    relu_pred_label = (relu_pred >= 0.15).astype(int)
    
    score = recall_score(y_test, relu_pred_label)
    recall_scores.append(score)
    print(f"Fold {i+1} Recall Score: {score:.4f}")
    
print(f"Average Recall Score: {sum(recall_scores)/len(recall_scores)}")

ValueError: Supported target types are: ('binary', 'multiclass'). Got 'continuous' instead.

**(h)** (15 points) Train and evaluate the `ElasticNet` regression model over the 5-fold cross-validation strategy defined in part (d). Ensure to standardize the inputs and optimize the values of `alpha` and `l1_ratio` first and then run the cross-validation experiment. Use the root mean squared error as measure of performance.

**(i)** (15 points) Train and evaluate the `GradientBoostingRegressor` model (`n_estimators=1000`, `max_depth=10`, and `random_state=42`) over the 5-fold cross-validation strategy defined in part (d). Use the root mean squared error as measure of performance.

In [ ]:

from sklearn.ensemble import GradientBoostingRegressor

GBR = GradientBoostingRegressor(n_estimators=1000, max_depth=10, random_state=42)

scores = -cross_val_score(GBR, x, y, cv=skf, scoring="neg_root_mean_squared_error")

print("RMSE per fold:", np.round(scores, 4))
print(f"Mean RMSE: {scores.mean():.4f} (± {scores.std():.4f})")


KeyboardInterrupt: 

**(j)** (3 points) Using the results from parts 1(e) to 1(i), what model would you use to predict `exam_score`?

_Your answer here._